Simple GenAI app  using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT")

In [3]:
### Data ingestion - we need to scrape the data from website
from langchain_community.document_loaders import WebBaseLoader

In [4]:
loader=WebBaseLoader("https://artincontext.org/urban-sketching/")
loader

In [5]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://artincontext.org/urban-sketching/', 'title': 'Urban Sketching - A Detailed Guide for Beginners', 'description': 'An Introduction to the Art of Urban Drawing ✔ An Easy Step-by-Step Drawing Tutorial ✔ Our Top Tips and Tricks ✔', 'language': 'en-US'}, page_content=' Urban Sketching - A Detailed Guide for Beginners               Skip to content  Search  Facebook Pinterest YouTube Instagram Linkedin TikTok Twitter HomeArt HistoryExpand ArtistsArtworksExpand PaintingsSculpturesArchitectureExpand ArchitectsArchitectural StylesBuildingsLiteratureExpand PoetryExpand Poetic TermsPoemsPoetsTypes of PoemsPhotographyExpand PhotographersPhotography TutorialsTypes of PhotographyPaintingExpand Color TheoryAcrylic PaintingWatercolor PaintingDrawingExpand Animal DrawingAnime DrawingDrawing TechniquesFood DrawingHuman Anatomy DrawingNature DrawingColoring PagesOther Art FormsExpand Art IndustryVideographyShopAbout  Toggle Menu    Search    Home / Drawing /Urban Sket

In [6]:
### Dividing the data into chunks--->text---->vectors---->vector embeddings---->vectorstore db
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter =RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [7]:
documents

[Document(metadata={'source': 'https://artincontext.org/urban-sketching/', 'title': 'Urban Sketching - A Detailed Guide for Beginners', 'description': 'An Introduction to the Art of Urban Drawing ✔ An Easy Step-by-Step Drawing Tutorial ✔ Our Top Tips and Tricks ✔', 'language': 'en-US'}, page_content='Urban Sketching - A Detailed Guide for Beginners               Skip to content  Search  Facebook Pinterest YouTube Instagram Linkedin TikTok Twitter HomeArt HistoryExpand ArtistsArtworksExpand PaintingsSculpturesArchitectureExpand ArchitectsArchitectural StylesBuildingsLiteratureExpand PoetryExpand Poetic TermsPoemsPoetsTypes of PoemsPhotographyExpand PhotographersPhotography TutorialsTypes of PhotographyPaintingExpand Color TheoryAcrylic PaintingWatercolor PaintingDrawingExpand Animal DrawingAnime DrawingDrawing TechniquesFood DrawingHuman Anatomy DrawingNature DrawingColoring PagesOther Art FormsExpand Art IndustryVideographyShopAbout  Toggle Menu    Search    Home / Drawing /Urban Sketc

In [8]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [9]:
from langchain_community.vectorstores import FAISS
vectorstoreDB=FAISS.from_documents(documents,embeddings)

In [10]:
vectorstoreDB

In [11]:
### Query from vectorstoreDB
query="How to create Urban Sketch?"
result=vectorstoreDB.similarity_search(query)
result[0].page_content

'improve your ability to draw objects in real life.\xa0Table of Contents ToggleAn Easy Guide to Urban SketchingNecessary MaterialsHow to Create Urban SketchesScouting\xa0PerspectiveLight and ShadowHow to Create an Urban SketchStep 1: Sketching the SceneStep 2: Building Up Layers of ColorStep 3: Enhancing Features With LineworkStep 4: Being Loose With Your DrawingTips and Tricks to RememberFrequently Asked QuestionsWhat Do You Need for Urban Sketching?How Do You Practice Urban Sketching?\xa0\xa0An Easy Guide to Urban SketchingUrban sketching seems a little daunting where you approach the drawing process without intention. However, there are many ways to pursue urban drawing for beginners. In this tutorial, we will look at some basic techniques for urban drawing that can improve your skills to draw objects from life. As we work through some simple drawing skills, you will find that the idea of urban sketching is quite a fun activity that can be done anywhere and at any time. \xa0Necessar

In [12]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [15]:
### Retrieval chain, Document chain
from langchain.chains.combine_documents import create_stuff_documents_chain 
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_messages([
    ("system",
     """
Answer the following questions based only on the provided context:
<context>
{context}
</context>
    """
),
    ("human","{input}")
])
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), config={'run_name': 'format_inputs'})
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template='\nAnswer the following questions based only on the provided context:\n<context>\n{context}\n</context>\n    ')), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001E2241238E0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E224123FD0>, root_client=<openai.OpenAI object at 0x000001E2241231F0>, root_async_client=<openai.AsyncOpenAI object at 0x000001E224123A60>, model_name='gpt-4o', openai_api_key=SecretStr('**********'), openai_proxy='')
| StrOutputParser(), config={'run_name': 'stuff_documents_

In [16]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":query,
    "context":[Document(page_content="Urban sketching is a form of art that involves drawing or painting on location, capturing the essence of a place in real-time. To create an urban sketch, you can follow these steps:")]
})

'To create an urban sketch, you can follow these steps:\n\n1. **Choose a Location**: Select a spot that inspires you and offers interesting subjects to sketch, such as buildings, streets, or people.\n\n2. **Gather Your Materials**: Bring along your sketchbook, pencils, pens, watercolors, or any other tools you prefer for sketching.\n\n3. **Find a Comfortable Spot**: Settle in a place where you have a good view of your chosen scene and where you can comfortably spend some time drawing.\n\n4. **Observe the Scene**: Take a moment to observe the details, the light, and the atmosphere of the location. Consider the composition and what elements you want to include.\n\n5. **Start Sketching**: Begin with light pencil outlines to capture the basic shapes and proportions. Focus on the overall structure before adding details.\n\n6. **Add Details and Values**: Gradually add more details and refine your sketch. Use shading and varying line weights to create depth and interest.\n\n7. **Incorporate C

In [ ]:
###Input---->Retriver---->vectorstoreDB
vectorstoreDB

In [19]:
retriever=vectorstoreDB.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [20]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E224120D90>), config={'run_name': 'retrieve_documents'})
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), config={'run_name': 'format_inputs'})
            | ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template='\nAnswer the following questions based only on the provided context:\n<context>\n{context}\n</context>\n    ')), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
            | ChatOpenAI(client=<openai.resources.chat.completions.completions.Comple

In [21]:
### get the response from the LLM
response=retrieval_chain.invoke({"input":query})
response['answer']

"Creating an urban sketch involves a series of steps that help you capture the essence of a scene in real life. Here's a general guide based on the provided context:\n\n1. **Sketching the Scene**: Begin by observing the scene you want to sketch. Look for interesting spaces, objects, or even people. Consider the perspective from which you will be viewing the scene and lightly sketch the outline.\n\n2. **Building Up Layers of Color**: If you're using color, start by adding light tonal values and gradually build up to darker tones. This can be done with pens, markers, or pencils, depending on your choice of materials.\n\n3. **Enhancing Features With Linework**: Use linework to highlight and define the features of your sketch. This can include the outlines of buildings, the details of objects, or the contours of shadows.\n\n4. **Being Loose With Your Drawing**: Allow yourself to be free and expressive with your sketching. Urban sketching is about capturing the moment and the essence of the

In [22]:
response

{'input': 'How to create Urban Sketch?',
 'context': [Document(metadata={'source': 'https://artincontext.org/urban-sketching/', 'title': 'Urban Sketching - A Detailed Guide for Beginners', 'description': 'An Introduction to the Art of Urban Drawing ✔ An Easy Step-by-Step Drawing Tutorial ✔ Our Top Tips and Tricks ✔', 'language': 'en-US'}, page_content='improve your ability to draw objects in real life.\xa0Table of Contents ToggleAn Easy Guide to Urban SketchingNecessary MaterialsHow to Create Urban SketchesScouting\xa0PerspectiveLight and ShadowHow to Create an Urban SketchStep 1: Sketching the SceneStep 2: Building Up Layers of ColorStep 3: Enhancing Features With LineworkStep 4: Being Loose With Your DrawingTips and Tricks to RememberFrequently Asked QuestionsWhat Do You Need for Urban Sketching?How Do You Practice Urban Sketching?\xa0\xa0An Easy Guide to Urban SketchingUrban sketching seems a little daunting where you approach the drawing process without intention. However, there ar

In [23]:
response['context']

[Document(metadata={'source': 'https://artincontext.org/urban-sketching/', 'title': 'Urban Sketching - A Detailed Guide for Beginners', 'description': 'An Introduction to the Art of Urban Drawing ✔ An Easy Step-by-Step Drawing Tutorial ✔ Our Top Tips and Tricks ✔', 'language': 'en-US'}, page_content='improve your ability to draw objects in real life.\xa0Table of Contents ToggleAn Easy Guide to Urban SketchingNecessary MaterialsHow to Create Urban SketchesScouting\xa0PerspectiveLight and ShadowHow to Create an Urban SketchStep 1: Sketching the SceneStep 2: Building Up Layers of ColorStep 3: Enhancing Features With LineworkStep 4: Being Loose With Your DrawingTips and Tricks to RememberFrequently Asked QuestionsWhat Do You Need for Urban Sketching?How Do You Practice Urban Sketching?\xa0\xa0An Easy Guide to Urban SketchingUrban sketching seems a little daunting where you approach the drawing process without intention. However, there are many ways to pursue urban drawing for beginners. In